# 02 — Training Replicas

For each window pair (A, B) trains **R XGBoost replicas** per window using stratified bootstrap sampling + different random seeds. Then evaluates on the common evaluation slice E_{A,B} and identifies the **flagged set** F_{A,B}.

**Input:** `data/processed/`, `data/windows/window_config.json`  
**Output per pair:** `data/models/pair_{i}/`
- `replicas_A/model_r{r}.joblib` (R models)
- `replicas_B/model_r{r}.joblib` (R models)
- `predictions.npz` — p̂_A, p̂_B, flagged indices, PR-AUC scores
- `scaler.joblib` — StandardScaler fitted on W_k

---

**Replica design (§3.1):** Each replica differs by:
1. A different random seed for the learning algorithm.
2. A stratified bootstrap sample of the training window (preserves class ratio).

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
from sklearn.metrics import average_precision_score
from xgboost import XGBClassifier

WORKSPACE  = Path(r'C:\Users\Frewl\OneDrive - KU Leuven\KU Leuven\Thesis\Shoppers_workspace')
PROC_DIR   = WORKSPACE / 'data' / 'processed'
WIN_DIR    = WORKSPACE / 'data' / 'windows'
MODEL_DIR  = WORKSPACE / 'data' / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ── XGBoost hyperparameters (fixed — no tuning needed) ──────────────────
XGB_PARAMS = dict(
    n_estimators    = 300,
    max_depth       = 6,
    learning_rate   = 0.05,
    subsample       = 0.8,
    colsample_bytree= 0.8,
    min_child_weight= 5,
    tree_method     = 'hist',   # fast histogram method
    eval_metric     = 'aucpr',
    verbosity       = 0,
    n_jobs          = -1,
)

print('Imports OK')

In [ ]:
# ── Load processed data ────────────────────────────────────────────────
X    = pd.read_parquet(PROC_DIR / 'X.parquet').values.astype(np.float32)
Y    = np.load(PROC_DIR / 'Y.npy').astype(np.int8)

with open(WIN_DIR / 'window_config.json') as f:
    config = json.load(f)

R   = config['parameters']['R']
TAU = config['parameters']['TAU']
pairs = config['pairs']

print(f'X: {X.shape}, Y: {Y.shape}')
print(f'R={R}, τ={TAU}, {len(pairs)} window pairs')

## Helper functions

In [ ]:
def stratified_bootstrap(idx: np.ndarray, Y: np.ndarray, seed: int) -> np.ndarray:
    """Stratified bootstrap: sample with replacement, preserving class ratio."""
    rng = np.random.default_rng(seed)
    pos = idx[Y[idx] == 1]
    neg = idx[Y[idx] == 0]
    n_pos = len(pos)
    n_neg = len(neg)
    boot_pos = rng.choice(pos, size=n_pos, replace=True)
    boot_neg = rng.choice(neg, size=n_neg, replace=True)
    return np.concatenate([boot_pos, boot_neg])


def train_replica(X_tr: np.ndarray, Y_tr: np.ndarray, seed: int) -> XGBClassifier:
    """Train one XGBoost replica with the given seed."""
    params = XGB_PARAMS.copy()
    params['random_state'] = seed
    # Handle class imbalance via scale_pos_weight
    n_neg = (Y_tr == 0).sum()
    n_pos = (Y_tr == 1).sum()
    if n_pos > 0:
        params['scale_pos_weight'] = n_neg / n_pos
    clf = XGBClassifier(**params)
    clf.fit(X_tr, Y_tr)
    return clf


def predict_proba_pos(model: XGBClassifier, X: np.ndarray) -> np.ndarray:
    """Return probability of positive class."""
    return model.predict_proba(X)[:, 1]


print('Helpers defined.')

## Main training loop

Skips pairs that have already been fully processed (checks for `predictions.npz`).

In [ ]:
SEED_BASE = 42   # seeds for replicas: SEED_BASE + r*100 + pair_id

performance_log = []   # collect PR-AUC results across pairs

for p in pairs:
    pid    = p['pair_id']
    pair_dir = MODEL_DIR / f'pair_{pid:02d}'
    pred_file = pair_dir / 'predictions.npz'

    if pred_file.exists():
        print(f'Pair {pid:02d}: already done, skipping.')
        data = np.load(pred_file)
        performance_log.append({
            'pair_id': pid,
            'pr_auc_A': float(data['pr_auc_A']),
            'pr_auc_B': float(data['pr_auc_B']),
            'n_flagged': int(data['flagged_idx'].shape[0]),
        })
        continue

    print(f'\n── Pair {pid:02d}: A_end={p["step_label_A"]}  B_end={p["step_label_B"]}  '
          f'eval={p["eval_start_label"]}→{p["eval_end_label"]}  '
          f'|A|={p["n_train_A"]:,} |B|={p["n_train_B"]:,} |eval|={p["n_eval"]:,} ──')

    idx_A    = np.array(p['idx_A'],    dtype=np.int64)
    idx_B    = np.array(p['idx_B'],    dtype=np.int64)
    idx_eval = np.array(p['idx_eval'], dtype=np.int64)

    X_eval = X[idx_eval]
    Y_eval = Y[idx_eval]

    # ── Fit scaler on W_k (window A) — reference scaler ─────────────
    scaler = StandardScaler()
    scaler.fit(X[idx_A])
    X_eval_sc = scaler.transform(X_eval)

    pair_dir.mkdir(parents=True, exist_ok=True)
    dir_A = pair_dir / 'replicas_A'
    dir_B = pair_dir / 'replicas_B'
    dir_A.mkdir(exist_ok=True)
    dir_B.mkdir(exist_ok=True)

    # ── Train R replicas for window A ────────────────────────────────
    preds_A = np.zeros((R, len(idx_eval)), dtype=np.float32)
    for r in range(R):
        seed = SEED_BASE + r * 100 + pid
        boot_idx = stratified_bootstrap(idx_A, Y, seed=seed)
        X_tr = scaler.transform(X[boot_idx])
        Y_tr = Y[boot_idx]
        model = train_replica(X_tr, Y_tr, seed=seed)
        preds_A[r] = predict_proba_pos(model, X_eval_sc)
        joblib.dump(model, dir_A / f'model_r{r}.joblib', compress=3)
        print(f'  A replica {r}: PR-AUC = {average_precision_score(Y_eval, preds_A[r]):.4f}')

    # ── Train R replicas for window B ────────────────────────────────
    preds_B = np.zeros((R, len(idx_eval)), dtype=np.float32)
    for r in range(R):
        seed = SEED_BASE + r * 100 + pid + 1000
        boot_idx = stratified_bootstrap(idx_B, Y, seed=seed)
        # Window B is also scaled with the W_k scaler (reference is fixed)
        X_tr = scaler.transform(X[boot_idx])
        Y_tr = Y[boot_idx]
        model = train_replica(X_tr, Y_tr, seed=seed)
        preds_B[r] = predict_proba_pos(model, X_eval_sc)
        joblib.dump(model, dir_B / f'model_r{r}.joblib', compress=3)
        print(f'  B replica {r}: PR-AUC = {average_precision_score(Y_eval, preds_B[r]):.4f}')

    # ── Aggregate predictions and compute flagged set ─────────────────
    p_hat_A = preds_A.mean(axis=0)   # replica-averaged probability
    p_hat_B = preds_B.mean(axis=0)

    # F_{A,B}: instances flagged by either model version
    flagged_mask = (p_hat_A >= TAU) | (p_hat_B >= TAU)
    flagged_local_idx = np.where(flagged_mask)[0]   # positions within idx_eval

    pr_auc_A = average_precision_score(Y_eval, p_hat_A)
    pr_auc_B = average_precision_score(Y_eval, p_hat_B)

    print(f'  Averaged: PR-AUC A = {pr_auc_A:.4f},  PR-AUC B = {pr_auc_B:.4f}')
    print(f'  Flagged: {flagged_local_idx.shape[0]:,} / {len(idx_eval):,} '
          f'({100*flagged_local_idx.shape[0]/len(idx_eval):.1f}%)')

    # Save scaler and predictions
    joblib.dump(scaler, pair_dir / 'scaler.joblib')
    np.savez_compressed(
        pred_file,
        preds_A      = preds_A,          # (R, n_eval)
        preds_B      = preds_B,          # (R, n_eval)
        p_hat_A      = p_hat_A,          # (n_eval,)
        p_hat_B      = p_hat_B,          # (n_eval,)
        flagged_idx  = flagged_local_idx, # local indices within idx_eval
        Y_eval       = Y_eval,
        pr_auc_A     = np.float32(pr_auc_A),
        pr_auc_B     = np.float32(pr_auc_B),
    )

    performance_log.append({
        'pair_id':  pid,
        'pr_auc_A': pr_auc_A,
        'pr_auc_B': pr_auc_B,
        'n_flagged': flagged_local_idx.shape[0],
    })

print('\n✓ All window pairs processed.')

## Performance summary

In [ ]:
perf_df = pd.DataFrame(performance_log)
print(perf_df.to_string(index=False))
print(f'\nMean PR-AUC A: {perf_df["pr_auc_A"].mean():.4f}')
print(f'Mean PR-AUC B: {perf_df["pr_auc_B"].mean():.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(perf_df['pair_id'], perf_df['pr_auc_A'], 'o-', label='Model A')
axes[0].plot(perf_df['pair_id'], perf_df['pr_auc_B'], 's--', label='Model B')
axes[0].set_title('PR-AUC over window pairs')
axes[0].set_xlabel('Window pair')
axes[0].set_ylabel('PR-AUC')
axes[0].legend()
axes[0].set_ylim(0, 1)

axes[1].bar(perf_df['pair_id'], perf_df['n_flagged'], color='tomato')
axes[1].set_title('Flagged instances per pair')
axes[1].set_xlabel('Window pair')
axes[1].set_ylabel('|F_{A,B}|')

plt.tight_layout()
plt.savefig(MODEL_DIR / 'performance_summary.png', dpi=120)
plt.show()

In [ ]:
# Save performance log
perf_df.to_csv(MODEL_DIR / 'performance_log.csv', index=False)
print('Performance log saved.')